# 🚀 MAGRPO Training on Kaggle

## Multi-Agent Reinforcement Learning for LLM Collaboration

Basé sur l'article : "LLM Collaboration with Multi-Agent Reinforcement Learning"

### Instructions :
1. **Uploadez vos checkpoints SFT** dans `/kaggle/input/checkpoints/`
2. **Uploadez votre dataset** dans `/kaggle/input/dataset/`
3. **Activez le GPU** dans les paramètres du notebook
4. **Exécutez toutes les cellules**


## 📦 Installation des Dépendances


In [ ]:
# Installation des packages nécessaires
%pip install -q transformers accelerate peft bitsandbytes datasets sentence-transformers


## ⚙️ Configuration


In [ ]:
import os
import json
import random
import torch
import logging
from typing import List, Dict, Any

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoModel
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

# Configuration pour Kaggle
BASE_MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Chemins Kaggle - Ajustez selon votre structure
CHECKPOINTS_DIR = "/kaggle/input/checkpoints"  # Dossier avec vos checkpoints SFT
DATASET_PATH = "/kaggle/input/dataset/orchestrator_sft.jsonl"  # Votre dataset
SAVE_FOLDER = "/kaggle/working/checkpoints/magrpo_rl"  # Sauvegarde dans working

AGENTS_LIST = ["orchestrator", "researcher", "code_writer", "critic"]
TOTAL_EPOCHS = 10  # Commencez avec 10 époques
SAVE_FREQ = 5  # Sauvegarder toutes les 5 époques
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
torch.cuda.empty_cache()

print(f"✅ Configuration prête")
print(f"   Device: {DEVICE}")
print(f"   Checkpoints: {CHECKPOINTS_DIR}")
print(f"   Dataset: {DATASET_PATH}")
print(f"   Save folder: {SAVE_FOLDER}")


## 🔧 Modules MAGRPO

### Centralized Critic


In [ ]:
class CentralizedCritic(nn.Module):
    """
    Simple MLP critic that takes global state embedding and outputs V(s).
    Keep it small for CPU training.
    """
    def __init__(self, input_dim: int = 384, hidden: int = 512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, 1)
        )

    def forward(self, state_emb: torch.Tensor):
        # state_emb: [batch, input_dim] or [input_dim]
        if state_emb.dim() == 1:
            state_emb = state_emb.unsqueeze(0)
        v = self.net(state_emb)
        return v.squeeze(-1)  # [batch] or scalar

print("✅ CentralizedCritic défini")


### Utilitaires MAGRPO


In [ ]:
# Lightweight sentence encoder for state embedding
STATE_ENCODER = "sentence-transformers/all-MiniLM-L6-v2"
_state_tokenizer = AutoTokenizer.from_pretrained(STATE_ENCODER)
_state_encoder = AutoModel.from_pretrained(STATE_ENCODER)

def encode_global_state(history_text: str, turn: int, current_agent: str, device: str = "cpu") -> torch.Tensor:
    """
    Encode S_t -> embedding tensor on CPU by default (small model).
    Returns tensor shape [emb_dim] (1D).
    """
    text = f"[TURN={turn}][AGENT={current_agent}] {history_text}"
    inputs = _state_tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = _state_encoder(**inputs)
        emb = outputs.last_hidden_state.mean(dim=1).squeeze(0)  # [emb_dim]
    return emb.to(device)

def compute_logprobs(model, input_ids: torch.Tensor, gen_ids: torch.Tensor, tokenizer, requires_grad: bool = False) -> torch.Tensor:
    """
    Compute sum log-probability of generated tokens under model.
    
    Args:
        model: The model to compute logprobs with
        input_ids: Input token IDs
        gen_ids: Generated token IDs
        tokenizer: Tokenizer
        requires_grad: If True, compute gradients (for actor). If False, no gradients (for ref model).
    """
    # Obtenir le device du modèle
    model_device = next(model.parameters()).device
    
    # S'assurer que le modèle est dans le bon mode
    if requires_grad:
        model.train()  # Mode training pour les gradients
    else:
        model.eval()  # Mode eval pour le ref model
    
    # S'assurer que les input_ids et gen_ids sont sur le même device que le modèle
    input_ids = input_ids.to(model_device)
    gen_ids = gen_ids.to(model_device)
    
    # Concatenate
    full = torch.cat([input_ids, gen_ids], dim=0).unsqueeze(0)  # [1, L_full]
    
    if requires_grad:
        # Pour l'actor, on a besoin des gradients - PAS de no_grad()
        outputs = model(full)
        logits = outputs.logits  # [1, L_full, V]
    else:
        # Pour le ref model, pas besoin de gradients
        with torch.no_grad():
            outputs = model(full)
            logits = outputs.logits  # [1, L_full, V]
    
    # Calculate log-probs
    L_in = input_ids.shape[0]
    L_gen = gen_ids.shape[0]
    lps = F.log_softmax(logits, dim=-1)  # [1, L_full, V]
    
    # Accumuler les log-probs (garder comme tensor pour les gradients)
    # Utiliser gather pour préserver les gradients
    if L_gen > 0:
        # Gather les log-probs pour chaque token généré
        # lps shape: [1, L_full, V]
        # On veut lps[0, pos-1, token_id] pour chaque pos, token_id
        selected_log_probs = []
        for i in range(L_gen):
            pos = L_in + i
            token_id = gen_ids[i].item()
            # Indexer correctement: lps[0, pos-1, token_id]
            selected_log_probs.append(lps[0, pos - 1, token_id])
        
        # Stack et sum pour préserver les gradients
        if len(selected_log_probs) > 0:
            total = torch.stack(selected_log_probs).sum()
        else:
            total = torch.tensor(0.0, device=model_device, requires_grad=requires_grad)
    else:
        total = torch.tensor(0.0, device=model_device, requires_grad=requires_grad)
    
    # S'assurer que le tensor est sur le même device que le modèle
    if total.device != model_device:
        total = total.to(model_device)
    
    return total  # scalar tensor

def move_model_to_device(model, device: str):
    try:
        model.to(device)
        torch.cuda.empty_cache()
    except Exception:
        pass

def offload_model_to_cpu(model):
    try:
        model.to("cpu")
        torch.cuda.empty_cache()
    except Exception:
        pass

print("✅ Utilitaires MAGRPO définis")


### MAGRPOTrainer


In [ ]:
class MAGRPOTrainer:
    """
    MAGRPO trainer for one agent.
    Uses shared critic (centralized) passed as 'critic' (on CPU).
    Actor/ref are PEFT models (4-bit on CPU) — moved to GPU during update.
    """
    def __init__(self, actor_model, ref_model, tokenizer, critic, lr: float = 1.41e-5, clip_epsilon: float = 0.2, device: str = "cuda"):
        self.actor = actor_model
        self.ref = ref_model
        self.tokenizer = tokenizer
        self.critic = critic  # this is on CPU
        self.clip_epsilon = clip_epsilon
        self.device = device

        # Only optimize LoRA parameters (PEFT exposes them)
        # S'assurer que les paramètres LoRA sont entraînables
        peft_params = [p for n,p in self.actor.named_parameters() if p.requires_grad]
        
        # Si aucun paramètre n'est entraînable, activer les paramètres LoRA
        if len(peft_params) == 0:
            print(f"⚠️  Aucun paramètre entraînable trouvé. Activation des paramètres LoRA...")
            # Activer les paramètres LoRA (PEFT utilise des noms spécifiques)
            for name, param in self.actor.named_parameters():
                # PEFT LoRA utilise: lora_A, lora_B, lora_embedding_A, etc.
                if any(keyword in name.lower() for keyword in ['lora', 'adapter', 'embedding']):
                    param.requires_grad = True
            peft_params = [p for n,p in self.actor.named_parameters() if p.requires_grad]
        
        # Vérification finale
        if len(peft_params) == 0:
            # Afficher les noms des paramètres pour debug
            param_names = [n for n, p in self.actor.named_parameters()]
            print(f"   ❌ Paramètres disponibles: {param_names[:10]}...")  # Afficher les 10 premiers
            raise ValueError(f"Aucun paramètre entraînable trouvé dans le modèle. Vérifiez que le modèle LoRA est correctement chargé.")
        
        # Compter le nombre total de paramètres entraînables
        trainable_count = sum(p.numel() for p in peft_params)
        print(f"   ✅ {len(peft_params)} groupes de paramètres LoRA ({trainable_count:,} paramètres totaux) pour l'optimisation")
        self.optimizer = Adam(peft_params, lr=lr)

    def compute_gae(self, rewards: List[float], values: List[float], gamma=0.99, lam=0.95):
        advs = []
        gae = 0.0
        values_ext = values + [0.0]
        for t in reversed(range(len(rewards))):
            delta = rewards[t] + gamma * values_ext[t+1] - values_ext[t]
            gae = delta + gamma * lam * gae
            advs.insert(0, gae)
        return advs

    def step(self, batch_queries: List[torch.Tensor], batch_responses: List[torch.Tensor], batch_rewards: List[float], batch_state_embs: List[torch.Tensor]):
        """
        batch_queries: list of input_ids 1D tensors (CPU)
        batch_responses: list of gen token 1D tensors (CPU)
        batch_rewards: list of floats
        batch_state_embs: list of state embeddings (CPU)
        """
        device = self.device

        # 1) compute values using critic (critic expected on CPU)
        values = []
        for emb in batch_state_embs:
            v = self.critic(emb.to("cpu")).item()
            values.append(v)

        # 2) compute advantages (GAE) using episodic rewards
        advantages = self.compute_gae(batch_rewards, values)
        advantages = torch.tensor(advantages, dtype=torch.float32)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        policy_losses = []
        kls = []
        # loop samples (safe offload model for each sample)
        for q_ids, gen_ids, adv in zip(batch_queries, batch_responses, advantages):
            # S'assurer que les tensors d'entrée sont sur le bon device
            q_ids = q_ids.to(device) if q_ids.device != torch.device(device) else q_ids
            gen_ids = gen_ids.to(device) if gen_ids.device != torch.device(device) else gen_ids
            adv_tensor = adv.to(device) if adv.device != torch.device(device) else adv
            
            # move models to GPU - IMPORTANT: s'assurer que TOUS les paramètres sont sur GPU
            move_model_to_device(self.actor, device)
            move_model_to_device(self.ref, device)
            
            # Vérifier que tous les paramètres entraînables sont sur GPU
            trainable_params = [p for p in self.actor.parameters() if p.requires_grad]
            if trainable_params:
                first_param_device = next(iter(trainable_params)).device
                if not str(first_param_device).startswith('cuda'):
                    print(f"   ⚠️  Paramètres entraînables sur {first_param_device}, déplacement vers {device}...")
                    self.actor.to(device)
            
            # S'assurer que l'actor est en mode training pour les gradients
            self.actor.train()
            self.ref.eval()  # Ref model en mode eval
            
            # Vérifier que les deux modèles sont sur le même device (GPU)
            actor_device = next(self.actor.parameters()).device
            ref_device = next(self.ref.parameters()).device
            
            # S'assurer que les deux modèles sont sur GPU
            if not str(actor_device).startswith('cuda'):
                print(f"   ⚠️  Actor sur {actor_device}, déplacement vers {device}...")
                move_model_to_device(self.actor, device)
                actor_device = next(self.actor.parameters()).device
            
            if not str(ref_device).startswith('cuda'):
                print(f"   ⚠️  Ref model sur {ref_device}, déplacement vers {device}...")
                move_model_to_device(self.ref, device)
                ref_device = next(self.ref.parameters()).device
            
            # S'assurer que les deux modèles sont sur le même device
            if actor_device != ref_device:
                print(f"   ⚠️  Devices différents: actor={actor_device}, ref={ref_device}. Normalisation...")
                move_model_to_device(self.ref, actor_device)
                ref_device = actor_device

            # compute old and new log probs
            # Ref model: pas de gradients (détaché) - calculé sur GPU
            with torch.no_grad():
                old_logp = compute_logprobs(self.ref, q_ids, gen_ids, self.tokenizer, requires_grad=False)
            
            # S'assurer que old_logp est sur le même device que le ref model
            if not isinstance(old_logp, torch.Tensor):
                old_logp = torch.tensor(float(old_logp), device=ref_device)
            elif old_logp.device != ref_device:
                old_logp = old_logp.to(ref_device)
            
            new_logp = compute_logprobs(self.actor, q_ids, gen_ids, self.tokenizer, requires_grad=True)
            
            # S'assurer que new_logp est sur le même device que l'actor
            if not isinstance(new_logp, torch.Tensor):
                new_logp = torch.tensor(float(new_logp), device=actor_device, requires_grad=True)
            elif new_logp.device != actor_device:
                new_logp = new_logp.to(actor_device)
            
            # S'assurer que requires_grad est activé
            if not new_logp.requires_grad:
                # Si pas de gradients, c'est un problème - vérifier le modèle
                print(f"   ⚠️  new_logp n'a pas de gradients. Vérification du modèle...")
                # Vérifier que le modèle est bien en mode training
                if not self.actor.training:
                    self.actor.train()
                # Recalculer si nécessaire
                new_logp = compute_logprobs(self.actor, q_ids, gen_ids, self.tokenizer, requires_grad=True)
                if not new_logp.requires_grad:
                    raise RuntimeError("Impossible d'obtenir des gradients pour new_logp. Vérifiez que les paramètres LoRA sont activés.")
            
            # Vérifier que new_logp a des gradients
            if not new_logp.requires_grad:
                print(f"   ⚠️  Attention: new_logp n'a pas de gradients. Vérifiez que le modèle actor est correctement configuré.")
                # Essayer de forcer les gradients en recalculant
                # Mais d'abord, vérifier que le modèle est bien en mode training
                if not self.actor.training:
                    self.actor.train()
                    print(f"   → Modèle actor mis en mode training")

            # Calculer le ratio (new_logp doit avoir des gradients)
            # S'assurer que tous les tensors sont sur le même device avant le calcul
            # Normaliser tous vers actor_device (qui est sur GPU)
            old_logp_detached = old_logp.detach()
            
            # Normaliser tous les tensors vers le même device (actor_device)
            if old_logp_detached.device != actor_device:
                old_logp_detached = old_logp_detached.to(actor_device)
            if new_logp.device != actor_device:
                new_logp = new_logp.to(actor_device)
            if adv_tensor.device != actor_device:
                adv_tensor = adv_tensor.to(actor_device)
            
            # Debug: vérifier les devices (après normalisation)
            devices_check = {
                'old_logp': str(old_logp_detached.device),
                'new_logp': str(new_logp.device),
                'adv_tensor': str(adv_tensor.device),
                'actor_device': str(actor_device)
            }
            
            # Vérification finale des devices
            # Tous les tensors doivent être sur le même device (actor_device)
            devices_list = [old_logp_detached.device, new_logp.device, adv_tensor.device]
            if any(str(d).startswith('cpu') for d in devices_list):
                print(f"   ❌ Problème: certains tensors sont sur CPU: {devices_check}")
                raise RuntimeError(f"Tensors sur CPU détectés: {devices_list}")
            
            # Vérifier que tous sont sur le même device
            unique_devices = set(str(d) for d in devices_list)
            if len(unique_devices) > 1:
                print(f"   ⚠️  Devices différents détectés: {unique_devices}")
                # Forcer tous vers actor_device
                old_logp_detached = old_logp_detached.to(actor_device)
                new_logp = new_logp.to(actor_device)
                adv_tensor = adv_tensor.to(actor_device)
            
            # Calculer le ratio et la loss (tous les tensors sont déjà sur actor_device)
            ratio = torch.exp(new_logp - old_logp_detached)
            unclipped = ratio * adv_tensor
            clipped = torch.clamp(ratio, 1.0 - self.clip_epsilon, 1.0 + self.clip_epsilon) * adv_tensor
            policy_loss = -torch.min(unclipped, clipped)
            
            # Vérifier que policy_loss est sur le même device que les autres tensors
            if policy_loss.device != actor_device:
                policy_loss = policy_loss.to(actor_device)
            
            # Vérifier que policy_loss a des gradients (nécessaire pour backward)
            if not policy_loss.requires_grad:
                # Si la loss n'a pas de gradients, c'est un problème critique
                print(f"   ❌ ERREUR: policy_loss n'a pas de gradients!")
                print(f"      new_logp.requires_grad: {new_logp.requires_grad}")
                print(f"      new_logp.device: {new_logp.device}")
                print(f"      ratio.requires_grad: {ratio.requires_grad if hasattr(ratio, 'requires_grad') else 'N/A'}")
                # Ne pas continuer si pas de gradients - cela indique un problème plus profond
                raise RuntimeError("policy_loss n'a pas de gradients. Vérifiez que new_logp a des gradients et que le modèle actor est en mode training.")

            policy_losses.append(policy_loss)
            # KL divergence (détaché car on ne veut pas de gradients pour la métrique)
            kls.append((old_logp.detach() - new_logp.detach()).cpu())

            # offload actor/ref immediately to free GPU before next sample
            offload_model_to_cpu(self.actor)
            offload_model_to_cpu(self.ref)

        if len(policy_losses) == 0:
            return {"loss": 0.0, "kl": 0.0, "value_mean": float(sum(values)/len(values) if values else 0.0)}

        # S'assurer que tous les policy_losses sont sur le même device
        # Obtenir le device du premier policy_loss (qui devrait être actor_device)
        if policy_losses:
            target_loss_device = policy_losses[0].device
            policy_losses = [pl.to(target_loss_device) if pl.device != target_loss_device else pl for pl in policy_losses]
        else:
            raise RuntimeError("Aucun policy_loss calculé!")
        
        loss = torch.stack(policy_losses).mean()
        # S'assurer que loss est sur le même device que les policy_losses
        if loss.device != target_loss_device:
            loss = loss.to(target_loss_device)
        
        # S'assurer que le modèle actor est sur le même device que la loss
        move_model_to_device(self.actor, target_loss_device)
        
        # Vérifier que tous les paramètres de l'optimizer sont sur le même device que la loss
        # avant le backward
        for param_group in self.optimizer.param_groups:
            for param in param_group['params']:
                if param.device != target_loss_device:
                    print(f"   ⚠️  Paramètre de l'optimizer sur {param.device}, déplacement vers {target_loss_device}...")
                    param.data = param.data.to(target_loss_device)
                    if param.grad is not None:
                        param.grad = param.grad.to(target_loss_device)
        
        # backward on PEFT params
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        return {
            "loss": loss.item(),
            "kl": float(torch.stack(kls).mean().item()) if kls else 0.0,
            "value_mean": float(sum(values) / len(values))
        }

print("✅ MAGRPOTrainer défini")


In [ ]:
def load_agent_policy(agent_name: str, is_training=True):
    lora_path = os.path.join(CHECKPOINTS_DIR, f"{agent_name}_lora")
    if not os.path.exists(os.path.join(lora_path, "adapter_config.json")):
        raise FileNotFoundError(f"Missing adapter for {agent_name} at {lora_path}")

    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, quantization_config=bnb, device_map="cpu")
    if is_training:
        prepare_model_for_kbit_training(base)
        base.config.use_cache = False
    else:
        base.config.use_cache = True

    lora_cfg = LoraConfig.from_pretrained(lora_path)
    model = get_peft_model(base, lora_cfg)
    
    # S'assurer que les paramètres LoRA sont entraînables si c'est pour l'entraînement
    if is_training:
        # Activer tous les paramètres LoRA pour l'entraînement
        # PEFT utilise des noms spécifiques comme 'lora_A', 'lora_B', etc.
        model.train()  # Mettre le modèle en mode training
        
        # Activer explicitement tous les paramètres LoRA
        for name, param in model.named_parameters():
            # Activer les paramètres LoRA (lora_A, lora_B, lora_embedding_A, etc.)
            if any(keyword in name.lower() for keyword in ['lora', 'adapter', 'embedding']):
                param.requires_grad = True
            else:
                param.requires_grad = False
        
        # Vérification
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in model.parameters())
        trainable_params_list = [n for n, p in model.named_parameters() if p.requires_grad]
        print(f"   📊 {lora_path.split('/')[-1]}: {trainable:,} paramètres entraînables sur {total:,} ({100*trainable/total:.2f}%)")
        if len(trainable_params_list) > 0:
            print(f"      Exemples de paramètres: {trainable_params_list[:3]}...")
    
    return model, tokenizer

print("✅ Fonction load_agent_policy définie")


### Classes Agent et Environnement


In [ ]:
# Simple prompt formatting
SYSTEM_PROMPTS = {
    "orchestrator": "Tu es l'Orchestrateur. REPLISSEZ JSON {'AGENT_CIBLE':..., 'COMMANDE':...}",
    "researcher": "Tu es le Researcher. Réponds factuellement.",
    "code_writer": "Tu es CodeWriter. Génère du code Python dans ```python```.",
    "critic": "Tu es Critic. Fais une critique concise."
}

def format_prompt(system_prompt, instruction):
    return f"<s>[INST] <<SYS>>\n{system_prompt}\n<</SYS>>\n\n{instruction} [/INST] "

class LLMAgent:
    def __init__(self, name):
        self.name = name
        self.system_prompt = SYSTEM_PROMPTS[name]
        self.model = None
        self.tokenizer = None

    def load_policy(self):
        self.model, self.tokenizer = load_agent_policy(self.name, is_training=True)
        # S'assurer que les paramètres LoRA sont activés
        trainable_count = sum(1 for p in self.model.parameters() if p.requires_grad)
        if trainable_count == 0:
            print(f"   ⚠️  Activation des paramètres LoRA pour {self.name}...")
            for param_name, param in self.model.named_parameters():
                if any(keyword in param_name.lower() for keyword in ['lora', 'adapter', 'embedding']):
                    param.requires_grad = True

    def generate_action(self, state_text):
        prompt = format_prompt(self.system_prompt, state_text)
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True).to("cuda")
        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=128)
        gen = out[0][inputs.input_ids.shape[1]:].detach().cpu()
        text = self.tokenizer.decode(gen, skip_special_tokens=True)
        return text, gen

class MARL_Env:
    def __init__(self, agents_list):
        self.agents = {}
        for name in agents_list:
            a = LLMAgent(name)
            a.load_policy()
            self.agents[name] = a
        self.current_state = ""
        self.current_agent = "orchestrator"
        self.turn_count = 0
        self.max_turns = 10

        self.last_query = ""
        self.last_response_tokens = None

    def reset(self, instr):
        self.current_state = f"Instruction: {instr}"
        self.current_agent = "orchestrator"
        self.turn_count = 0
        return self.current_state

    def step(self):
        agent_name = self.current_agent
        agent = self.agents[agent_name]

        # offload others
        for n,a in self.agents.items():
            if n != agent_name:
                offload_model_to_cpu(a.model)

        # move actor to gpu
        move_model_to_device(agent.model, "cuda")

        # store prompt used
        self.last_query = format_prompt(agent.system_prompt, self.current_state)
        text, gen = agent.generate_action(self.current_state)
        self.last_response_tokens = gen  # CPU tensor

        # offload actor
        offload_model_to_cpu(agent.model)

        # transition logic
        reward = 0.0
        done = False
        info = {"response": text}

        if agent_name == "orchestrator":
            try:
                j = json.loads(text)
                tgt = j.get("AGENT_CIBLE","").lower()
                cmd = j.get("COMMANDE","")
                if tgt == "end":
                    done = True
                    reward = 5.0
                elif tgt in self.agents:
                    self.current_agent = tgt
                    self.current_state += f"\n[ORCH->{tgt}]: {cmd}"
                else:
                    done = True
                    reward = -3.0
            except:
                done = True
                reward = -5.0
        else:
            self.current_state += f"\n[{agent_name.upper()}]: {text}"
            self.current_agent = "orchestrator"

        self.turn_count += 1
        if self.turn_count >= self.max_turns:
            done = True
            reward = -5.0

        return self.current_state, reward, done, info

print("✅ Classes Agent et Environnement définies")


### Collection de Trajectoires


In [ ]:
def collect_trajectories(env: MARL_Env, dataset, max_episodes: int):
    trajs = []
    num = min(max_episodes, len(dataset))
    for _ in range(num):
        idx = random.randint(0, len(dataset)-1)
        instr = dataset[idx]["instruction"]
        env.reset(instr)
        episode_steps = []
        done = False
        final_reward = 0.0
        while not done:
            agent_name = env.current_agent
            # encode state
            state_emb = encode_global_state(env.current_state, env.turn_count, agent_name, device="cpu")
            new_state, r, done, info = env.step()
            if info.get("response") is not None:
                # get query tokens and response tokens
                agent = env.agents[agent_name]
                q_ids = agent.tokenizer(env.last_query, return_tensors="pt", truncation=True).input_ids.squeeze(0).cpu()
                resp_ids = env.last_response_tokens.cpu() if env.last_response_tokens is not None else torch.tensor([], dtype=torch.long)
                episode_steps.append({
                    "agent": agent_name,
                    "query": q_ids,
                    "response": resp_ids,
                    "state_emb": state_emb.detach().cpu()
                })
            if done:
                final_reward = r
                for s in episode_steps:
                    trajs.append({
                        "agent": s["agent"],
                        "query": s["query"],
                        "response": s["response"],
                        "reward": float(final_reward),
                        "state_emb": s["state_emb"]
                    })
                break
    logging.info(f"Collected {len(trajs)} transitions.")
    return trajs

print("✅ Fonction collect_trajectories définie")


## 🚀 Entraînement MAGRPO


In [ ]:
def train_marl_magrpo(agents_list):
    print("\n" + "="*70)
    print("🚀 DÉMARRAGE DE L'ENTRAÎNEMENT MAGRPO")
    print("="*70)
    
    # Initialiser l'environnement
    print("\n📦 Initialisation de l'environnement...")
    env = MARL_Env(agents_list)
    
    # Vérifier que tous les modèles ont des paramètres entraînables
    print("\n🔍 Vérification des paramètres entraînables...")
    for name in agents_list:
        trainable_params = [p for n, p in env.agents[name].model.named_parameters() if p.requires_grad]
        print(f"   {name}: {len(trainable_params)} paramètres entraînables")
        if len(trainable_params) == 0:
            print(f"   ⚠️  Aucun paramètre entraînable pour {name}. Activation...")
            for param_name, param in env.agents[name].model.named_parameters():
                if any(keyword in param_name.lower() for keyword in ['lora', 'adapter', 'embedding']):
                    param.requires_grad = True
            trainable_params = [p for n, p in env.agents[name].model.named_parameters() if p.requires_grad]
            print(f"   ✅ {len(trainable_params)} paramètres activés pour {name}")
    
    # Charger le dataset
    print(f"\n📊 Chargement du dataset depuis {DATASET_PATH}...")
    dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
    print(f"✅ Dataset chargé: {len(dataset)} échantillons")
    
    # Instancier le critic centralisé
    print("\n🎯 Initialisation du critic centralisé...")
    state_dim = 384
    critic = CentralizedCritic(input_dim=state_dim, hidden=512)
    critic.to("cpu")
    critic_opt = torch.optim.Adam(critic.parameters(), lr=1e-4)
    
    # Préparer les trainers pour chaque agent
    print("\n🤖 Préparation des trainers pour chaque agent...")
    trainers = {}
    for name in agents_list:
        print(f"   Chargement de {name}...")
        actor = env.agents[name].model
        
        # Vérifier que l'actor a des paramètres entraînables
        trainable_params = [p for n, p in actor.named_parameters() if p.requires_grad]
        if len(trainable_params) == 0:
            print(f"   ⚠️  Activation des paramètres LoRA pour {name}...")
            for param_name, param in actor.named_parameters():
                if any(keyword in param_name.lower() for keyword in ['lora', 'adapter', 'embedding']):
                    param.requires_grad = True
        
        ref_model, _ = load_agent_policy(name, is_training=False)
        offload_model_to_cpu(ref_model)
        
        try:
            trainer = MAGRPOTrainer(actor, ref_model, env.agents[name].tokenizer, critic, lr=1.41e-5, clip_epsilon=0.2, device="cuda")
            trainers[name] = trainer
            logging.info(f"Trainer ready for {name}")
        except ValueError as e:
            print(f"   ❌ Erreur lors de la création du trainer pour {name}: {e}")
            print(f"   Vérifiez que le modèle LoRA est correctement chargé.")
            raise
    
    print("\n" + "="*70)
    print("🏋️ DÉBUT DE L'ENTRAÎNEMENT")
    print("="*70)
    
    # Boucle d'entraînement
    for epoch in range(TOTAL_EPOCHS):
        print(f"\n{'='*70}")
        print(f"📅 ÉPOQUE {epoch + 1}/{TOTAL_EPOCHS}")
        print(f"{'='*70}")
        
        # Collecter des trajectoires
        print("\n🔄 Collection de trajectoires...")
        transitions = collect_trajectories(env, dataset, max_episodes=2)  # small for T4
        
        if not transitions:
            logging.warning("No transitions collected.")
            break
        
        # Grouper par agent
        batches = {n: {"query":[], "response":[], "reward":[], "state":[]} for n in agents_list}
        for t in transitions:
            batches[t["agent"]]["query"].append(t["query"])
            batches[t["agent"]]["response"].append(t["response"])
            batches[t["agent"]]["reward"].append(t["reward"])
            batches[t["agent"]]["state"].append(t["state_emb"])
        
        # Mettre à jour chaque agent
        print("\n📈 Mise à jour des agents...")
        for name in agents_list:
            b = batches[name]
            if not b["query"]:
                continue
            stats = trainers[name].step(b["query"], b["response"], b["reward"], b["state"])
            print(f"   {name:15s}: loss={stats['loss']:.4f}, kl={stats['kl']:.6f}, val_mean={stats['value_mean']:.3f}")
            logging.info(f"{name} update: loss {stats['loss']:.4f} kl {stats['kl']:.6f} val_mean {stats['value_mean']:.3f}")
        
        # Sauvegarder les checkpoints
        if (epoch + 1) % SAVE_FREQ == 0:
            print(f"\n💾 Sauvegarde des checkpoints (époque {epoch + 1})...")
            os.makedirs(SAVE_FOLDER, exist_ok=True)
            for name in agents_list:
                save_path = os.path.join(SAVE_FOLDER, f"epoch{epoch+1}_{name}_rl")
                try:
                    env.agents[name].model.save_pretrained(save_path)
                    print(f"   ✅ {name} sauvegardé -> {save_path}")
                    logging.info(f"Saved RL LoRA for {name} -> {save_path}")
                except Exception as e:
                    print(f"   ❌ Échec sauvegarde {name}: {e}")
                    logging.warning(f"Failed to save {name}: {e}")
    
    print("\n" + "="*70)
    print("✅ ENTRAÎNEMENT TERMINÉ")
    print("="*70)
    logging.info("Training finished.")

print("✅ Fonction train_marl_magrpo définie")


## ▶️ Lancer l'Entraînement


In [ ]:
# Vérifier que les checkpoints existent
print("🔍 Vérification des checkpoints SFT...")
for agent in AGENTS_LIST:
    checkpoint_path = os.path.join(CHECKPOINTS_DIR, f"{agent}_lora")
    if os.path.exists(checkpoint_path):
        print(f"   ✅ {agent}: {checkpoint_path}")
    else:
        print(f"   ❌ {agent}: MANQUANT à {checkpoint_path}")
        print(f"      ⚠️  Assurez-vous d'avoir uploadé les checkpoints SFT dans /kaggle/input/checkpoints/")

print("\n" + "="*70)
print("🚀 LANCEMENT DE L'ENTRAÎNEMENT")
print("="*70)

# Lancer l'entraînement
train_marl_magrpo(AGENTS_LIST)


## 📦 Sauvegarder les Résultats


In [ ]:
# Les checkpoints sont automatiquement sauvegardés dans /kaggle/working/
# Vous pouvez les télécharger depuis l'onglet Output du notebook

print("📁 Checkpoints sauvegardés dans:")
print(f"   {SAVE_FOLDER}")
print("\n💡 Pour télécharger:")
print("   1. Allez dans l'onglet 'Output' du notebook")
print("   2. Téléchargez le dossier 'checkpoints/magrpo_rl'")

# Lister les fichiers sauvegardés
if os.path.exists(SAVE_FOLDER):
    print("\n📋 Fichiers sauvegardés:")
    for root, dirs, files in os.walk(SAVE_FOLDER):
        level = root.replace(SAVE_FOLDER, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 2 * (level + 1)
        for file in files:
            print(f"{subindent}{file}")
